**TP 1INF3721 :**

**Les membres du GROUPE 3**

*   **NASSARAMADJI NASSAIRE 23**
*   **HAMIT ALI MAHAMAT 21T2884**
*   **MARIE**






<br><br><br>


<br><br><br>


**Etape 1 : Préparation du jeu de données**

**1 - téléchargement de données de tous les pays**

<br><br><br>


Au lieu de télécharger les données 217 pays manuellement , nous avons écrit un script permettant de télécharger le dossier zippé de tous les pays , chaque dossier contient :

*   Metadata_Indicator_API_
*   Metadata_Country_API_
*   Un fichier de préfixe API_

Puis le script les décompresse dans un seul dossier , nous avons :

*   **3 fichiers pour 217 pays**
*   Au total **651 fichiers** dans notre dossier **donnees_banque_mondiale**








In [ ]:
import os
import requests
import zipfile
import io
import shutil
from google.colab import files

def telecharger_et_zipper_donnees():
    # 1. Création du dossier cible
    dossier_destination = "donnees_banque_mondiale"
    os.makedirs(dossier_destination, exist_ok=True)
    print(f"Dossier '{dossier_destination}' prêt.")

    # 2. Récupération des codes des 217 pays
    print("Récupération de la liste des pays via l'API...")
    url_pays = "https://api.worldbank.org/v2/country?format=json&per_page=300"
    reponse_pays = requests.get(url_pays)
    donnees_json = reponse_pays.json()

    # Filtrer les agrégats pour ne garder que les pays réels
    codes_pays = [pays['id'] for pays in donnees_json[1] if pays['region']['id'] != 'NA']
    print(f"{len(codes_pays)} pays identifiés. Début des téléchargements et de l'extraction...")

    # 3. Téléchargement et extraction (les 651 fichiers iront dans le dossier)
    for code_pays in codes_pays:
        url_archive = f"https://api.worldbank.org/v2/fr/country/{code_pays}?downloadformat=csv"
        try:
            reponse_telechargement = requests.get(url_archive)

            # Vérifier si le téléchargement a réussi (code HTTP 200 = succès)
            if reponse_telechargement.status_code == 200:
                # Lecture en mémoire et extraction des 3 CSV directement dans le dossier
                fichier_zip = zipfile.ZipFile(io.BytesIO(reponse_telechargement.content))
                fichier_zip.extractall(dossier_destination)
            else:
                print(f"❌ Erreur de téléchargement pour le pays {code_pays}")

        except Exception as erreur:
            print(f"❌ Échec pour {code_pays} : {erreur}")

    print("\n✅ Extraction terminée ! Les 651 fichiers (3 par pays) sont dans le dossier.")

    # 4. Compression du dossier en un fichier ZIP (méthode 100% Python)
    print("Compression du dossier en cours...")
    # shutil.make_archive(nom_archive_sortie, format, dossier_a_compresser)
    shutil.make_archive(dossier_destination, 'zip', dossier_destination)
    print(f"✅ Fichier '{dossier_destination}.zip' créé avec succès.")

    # 5. Déclenchement du téléchargement vers votre ordinateur
    print("Lancement du téléchargement...")
    files.download(f'{dossier_destination}.zip')

# Exécution du programme complet
telecharger_et_zipper_donnees()

📁 Dossier 'donnees_banque_mondiale' prêt.
🌍 Récupération de la liste des pays via l'API...
⏳ 217 pays identifiés. Début des téléchargements et de l'extraction...

✅ Extraction terminée ! Les 651 fichiers (3 par pays) sont dans le dossier.
📦 Compression du dossier en cours...
✅ Fichier 'donnees_banque_mondiale.zip' créé avec succès.
⬇️ Lancement du téléchargement...


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<br><br><br>


<br><br><br>


**Etape 1 : Préparation du jeu de données**

**2 : Intégration de données**

<br><br><br>


Après avoir téléchager les données de tous les pays , nous procedons à les intégrer pour avoir un seul dataset de :

*   217 observations (lignes )
*   1517 attributs ( colonnes)

Pour cela, nous avons écrit un script qui permet:

*  d'extraire uniquement **l'année 2023**  de chaque **fichier API_**,

*  de transposer les lignes en colonnes,

* et d'y associer la classe **Income_Group** issue des fichiers **Metadata_Country_**



Nous avons besoin des bibliothèques suivantes et nous avons importer notre dossier contenant 651 fichiers  

En suite ,

nous bouclons sur chaque fichier **pays**

 Nous extrayons l'année 2023, et nous utilisons la fonction **pivot pour transformer les indicateurs (qui sont en lignes) en colonnes**, afin qu'ils deviennent nos attributs.


En parallèle, nous récupérons la classe **(IncomeGroup)** dans le fichier de métadonnées associé

In [17]:
import os
import glob
import pandas as pd

dossier_source = "donnees_banque_mondiale"
fichiers_donnees = glob.glob(os.path.join(dossier_source, "API_*.csv"))

print("🔄 Début de l'extraction des données...")

liste_tableaux_indicateurs = []
liste_tableaux_classes = []

for chemin in fichiers_donnees:
    # --- ÉTAPE A : Extraire les attributs ---
    df_api = pd.read_csv(chemin, skiprows=4)
    df_filtre = df_api[['Country Code', 'Indicator Code', '2023']]
    df_pivot = df_filtre.pivot(index='Country Code', columns='Indicator Code', values='2023').reset_index()
    liste_tableaux_indicateurs.append(df_pivot)

    # --- ÉTAPE B : Extraire la classe ---
    nom_fichier_meta = f"Metadata_Country_{os.path.basename(chemin)}"
    chemin_meta = os.path.join(dossier_source, nom_fichier_meta)

    if os.path.exists(chemin_meta):
        df_meta = pd.read_csv(chemin_meta)
        df_meta.columns = df_meta.columns.str.strip()

        if 'Country Code' in df_meta.columns and 'Income_Group' in df_meta.columns:
            df_meta_filtre = df_meta[['Country Code', 'Income_Group']].copy()
            df_meta_filtre = df_meta_filtre.dropna(subset=['Income_Group'])

            # On la nomme 'IncomeGroup' ici pour que ta cellule d'assemblage fonctionne parfaitement
            df_meta_filtre.rename(columns={'Income_Group': 'IncomeGroup'}, inplace=True)
            liste_tableaux_classes.append(df_meta_filtre)

print(f"✅ Extraction terminée ! {len(liste_tableaux_indicateurs)} pays traités.")

🔄 Début de l'extraction des données...
✅ Extraction terminée ! 217 pays traités.


<br><br><br>


<br><br><br>


**Etape 1 : Préparation du jeu de données**

3 - Concaténation et fusion






  Maintenant que nous avons des listes contenant les données de chaque pays, nous les empilons (concat) pour créer deux grands tableaux : un pour les attributs et un pour les classes. Enfin, nous fusionnons (merge) ces deux tableaux en utilisant le code du pays comme clé de jointure





In [18]:
print("Assemblage des matrices en cours...")

# 1. Concaténer (empiler) les données de tous les pays
matrice_indicateurs = pd.concat(liste_tableaux_indicateurs, ignore_index=True)

# drop_duplicates assure de ne garder qu'une seule fois la classe par pays
matrice_classes = pd.concat(liste_tableaux_classes, ignore_index=True).drop_duplicates()

# 🔥 LE CORRECTIF : Nettoyer les espaces invisibles avant de renommer et fusionner
matrice_indicateurs['Country Code'] = matrice_indicateurs['Country Code'].astype(str).str.strip()
matrice_classes['Country Code'] = matrice_classes['Country Code'].astype(str).str.strip()

# 2. Renommer les colonnes pour respecter la nomenclature de l'énoncé
matrice_indicateurs.rename(columns={'Country Code': 'Country_code'}, inplace=True)
matrice_classes.rename(columns={'Country Code': 'Country_code', 'IncomeGroup': 'Income_Group'}, inplace=True)

# 3. Fusionner les attributs et la classe grâce à la clé "Country_code"
jeu_de_donnees_final = pd.merge(matrice_indicateurs, matrice_classes, on='Country_code', how='inner')

print("✅ Fusion terminée ! Le jeu de données global est assemblé.")

Assemblage des matrices en cours...
✅ Fusion terminée ! Le jeu de données global est assemblé.


<br><br><br>


<br><br><br>


**Etape 1 : Préparation du jeu de données**

4 - Vérification et Sauvegarde


Pour terminer, nous vérifions que les dimensions de notre matrice correspondent aux exigences du TP (217 pays et 1516 attributs + la classe). Ensuite, nous générons le fichier CSV final et le téléchargeons





In [19]:
from google.colab import files

# 1. Affichage des dimensions de la matrice
lignes, colonnes = jeu_de_donnees_final.shape
print(f"Dimensions du jeu de données : {lignes} observations, {colonnes} colonnes.")

# Afficher les 5 premières lignes pour vérifier visuellement
display(jeu_de_donnees_final.head())

# 2. Sauvegarder et déclencher le téléchargement automatique
nom_fichier_final = "dataset_group3.csv"
jeu_de_donnees_final.to_csv(nom_fichier_final, index=False)

print(f"Fichier généré. Lancement du téléchargement vers votre ordinateur...")
files.download(nom_fichier_final)

Dimensions du jeu de données : 217 observations, 1518 colonnes.


,Country_code,AG.CON.FERT.PT.ZS,AG.CON.FERT.ZS,AG.LND.AGRI.K2,AG.LND.AGRI.ZS,AG.LND.ARBL.HA,AG.LND.ARBL.HA.PC,AG.LND.ARBL.ZS,AG.LND.CREL.HA,AG.LND.CROP.ZS,...,per_sa_allsa.cov_q5_tot,per_si_allsi.adq_pop_tot,per_si_allsi.ben_q1_tot,per_si_allsi.cov_pop_tot,per_si_allsi.cov_q1_tot,per_si_allsi.cov_q2_tot,per_si_allsi.cov_q3_tot,per_si_allsi.cov_q4_tot,per_si_allsi.cov_q5_tot,Income_Group
0,LBY,252.733333,22.040698,153500.0,8.723871,1720000.0,0.235434,0.977528,309742.0,0.187549,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,"Revenu intermédiaire, tranche supérieure"
1,CPV,NaN,3.540000,790.0,19.602978,50000.0,0.095725,12.406948,20190.0,0.992556,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,"Revenu intermédiaire, tranche supérieure"
2,YEM,NaN,8.930052,234520.0,44.419190,1158000.0,0.029398,2.193306,543000.0,0.556850,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Faible revenu
3,SLE,NaN,2.666035,39490.0,54.710446,1584000.0,0.187223,21.945137,1045605.0,2.285952,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Faible revenu
4,ISL,NaN,97.289256,16380.0,16.245165,121000.0,0.313745,1.200040,3160.0,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Revenu élevé


Fichier généré. Lancement du téléchargement vers votre ordinateur...


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<br><br><br>


<br><br><br>


In [ ]:
import os
import glob
import pandas as pd
from google.colab import files

def construire_dataset_final():
    dossier_source = "donnees_banque_mondiale"
    fichiers_donnees = glob.glob(os.path.join(dossier_source, "API_*.csv"))

    print(f"🔄 Début du traitement de {len(fichiers_donnees)} fichiers pays...")

    liste_indicateurs = []
    liste_classes = []

    for chemin in fichiers_donnees:
        # --- 1. Extraire les attributs (Fichiers API_) ---
        # skiprows=4 permet d'ignorer les 4 premières lignes d'en-tête
        df_api = pd.read_csv(chemin, skiprows=4)

        # Sélectionner le code pays, le code indicateur et l'année 2023
        df_filtre = df_api[['Country Code', 'Indicator Code', '2023']]

        # Pivoter pour que les 1516 indicateurs deviennent des colonnes
        df_pivot = df_filtre.pivot(index='Country Code', columns='Indicator Code', values='2023').reset_index()
        liste_indicateurs.append(df_pivot)

        # --- 2. Extraire la classe (Fichiers Metadata_Country_) ---
        nom_fichier_meta = f"Metadata_Country_{os.path.basename(chemin)}"
        chemin_meta = os.path.join(dossier_source, nom_fichier_meta)

        if os.path.exists(chemin_meta):
            df_meta = pd.read_csv(chemin_meta)

            # Nettoyer les noms des colonnes pour éviter tout espace parasite
            df_meta.columns = df_meta.columns.str.strip()

            # Extraire uniquement le Country Code et Income_Group
            if 'Country Code' in df_meta.columns and 'Income_Group' in df_meta.columns:
                df_meta_filtre = df_meta[['Country Code', 'Income_Group']].copy()

                # Ignorer les pays qui n'ont pas de groupe de revenu classifié
                df_meta_filtre = df_meta_filtre.dropna(subset=['Income_Group'])
                liste_classes.append(df_meta_filtre)

    print("🔗 Assemblage des données en cours...")

    # Concaténer les listes en deux grands tableaux
    matrice_indicateurs = pd.concat(liste_indicateurs, ignore_index=True)
    matrice_classes = pd.concat(liste_classes, ignore_index=True).drop_duplicates()

    # 🔥 LE CORRECTIF : Nettoyer les espaces invisibles dans les codes pays avant la fusion
    matrice_indicateurs['Country Code'] = matrice_indicateurs['Country Code'].astype(str).str.strip()
    matrice_classes['Country Code'] = matrice_classes['Country Code'].astype(str).str.strip()

    # --- 3. Fusion finale ---
    # On croise les attributs et la classe grâce au code pays
    dataset = pd.merge(matrice_indicateurs, matrice_classes, on='Country Code', how='inner')

    # Renommer les colonnes selon les consignes exactes de l'énoncé du TP
    dataset.rename(columns={'Country Code': 'Country_code'}, inplace=True)

    # --- 4. Affichage et Sauvegarde ---
    print("\n✅ Dataset alimenté avec succès !")
    print(f"📊 Dimensions finales : {dataset.shape[0]} observations (pays) et {dataset.shape[1]} colonnes.")

    nom_fichier_final = "dataset_INF372_final.csv"
    dataset.to_csv(nom_fichier_final, index=False)

    print("⬇️ Lancement du téléchargement vers votre ordinateur...")
    files.download(nom_fichier_final)

    return dataset

# Exécution du programme
dataset_final = construire_dataset_final()

# Afficher les 5 premières lignes pour vérifier
display(dataset_final.head())

🔄 Début du traitement de 217 fichiers pays...
🔗 Assemblage des données en cours...

✅ Dataset alimenté avec succès !
📊 Dimensions finales : 217 observations (pays) et 1518 colonnes.
⬇️ Lancement du téléchargement vers votre ordinateur...


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

,Country_code,AG.CON.FERT.PT.ZS,AG.CON.FERT.ZS,AG.LND.AGRI.K2,AG.LND.AGRI.ZS,AG.LND.ARBL.HA,AG.LND.ARBL.HA.PC,AG.LND.ARBL.ZS,AG.LND.CREL.HA,AG.LND.CROP.ZS,...,per_sa_allsa.cov_q5_tot,per_si_allsi.adq_pop_tot,per_si_allsi.ben_q1_tot,per_si_allsi.cov_pop_tot,per_si_allsi.cov_q1_tot,per_si_allsi.cov_q2_tot,per_si_allsi.cov_q3_tot,per_si_allsi.cov_q4_tot,per_si_allsi.cov_q5_tot,Income_Group
0,LBY,252.733333,22.040698,153500.0,8.723871,1720000.0,0.235434,0.977528,309742.0,0.187549,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,"Revenu intermédiaire, tranche supérieure"
1,CPV,NaN,3.540000,790.0,19.602978,50000.0,0.095725,12.406948,20190.0,0.992556,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,"Revenu intermédiaire, tranche supérieure"
2,YEM,NaN,8.930052,234520.0,44.419190,1158000.0,0.029398,2.193306,543000.0,0.556850,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Faible revenu
3,SLE,NaN,2.666035,39490.0,54.710446,1584000.0,0.187223,21.945137,1045605.0,2.285952,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Faible revenu
4,ISL,NaN,97.289256,16380.0,16.245165,121000.0,0.313745,1.200040,3160.0,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Revenu élevé
